In [ ]:
import cv2
import einops
import matplotlib.pyplot as plt
import mediapy
import numpy as np

import os

from openpi.policies.libero_reason_dataset import LiberoSkillReasonDataset
from openpi.training import config as _config

In [ ]:
data_config = _config.get_config('pi05_libero_skill_reason_fixed')
dataset = LiberoSkillReasonDataset(data_config.data.base_config, data_config.model.action_horizon)

In [ ]:
import os
from pathlib import Path
import sys
SCRIPT_DIR = Path("../py_script")
sys.path.append(str(SCRIPT_DIR))
from vlm_interfaces import *

In [ ]:
import importlib
import vla_verify
importlib.reload(vla_verify.scene_graph)
from vla_verify.scene_graph import TaskSceneGraph
from vla_verify.verifier import VLAVerifier
PDDL_PATH = SCRIPT_DIR / "pddl" / "libero_domain.pddl"
pddl_domain_text = open(PDDL_PATH).read()

llm_interface, vlm_interface = get_openrouter_interfaces()
scene_graph = TaskSceneGraph(pddl_domain_text, vlm_interface)
verifier = VLAVerifier(scene_graph, llm_interface)

In [ ]:
def image_tensor_to_cv2(image, resolution=(512,512)):
    return cv2.resize(np.array(einops.rearrange(image, "c h w -> h w c") * 255, dtype=np.uint8), resolution, interpolation=cv2.INTER_LANCZOS4)

def get_episode(episode_idx):
    reasonings = dataset.reasoning[episode_idx]
    start_idx = dataset.episode_starts[episode_idx]
    end_idx = dataset.episode_ends[episode_idx]
    data = dataset.hf_dataset[int(start_idx)]
    video_frames = []
    for i in range(start_idx, end_idx):
        img_data = dataset.hf_dataset[i]['image']
        video_frames.append(image_tensor_to_cv2(img_data))
    return reasonings, video_frames

print(len(dataset.episode_starts))
reasonings, video_frames = get_episode(391)
print(reasonings['instruction'])
mediapy.write_video(f'sample.mp4', video_frames, fps=20)

In [ ]:
out_dir = "data"
import gzip
import os
import re
import json
import scipy
os.makedirs(out_dir, exist_ok=True)

def process_episode(episode, episode_idx):
    reasonings, video_frames = episode
    task = reasonings['instruction'].split(':', 1)[-1].strip()
    print(task)
    scene_graph.read_image(video_frames[0], hint=f"The robot is trying to {task}", ground=False)

    targets = []
    for segment in reasonings['segments']:
        skill = segment['skill']
        splits = re.split('([^a-zA-Z0-9]left[^a-zA-Z0-9]|[^a-zA-Z0-9]right[^a-zA-Z0-9])', skill)
        for i in range(1, len(splits), 2):
            if len(splits[i]) == 6:
                splits[i] = splits[i][0] + 'right' + splits[i][-1]
            elif len(splits[i]) == 7:
                splits[i] = splits[i][0] + 'left' + splits[i][-1]
        skill = ''.join(splits)
        skill_res = verifier.verify_skill(skill)
        print(skill_res)
        if not skill_res.accepted:
            print("Verification failed!")
            break
        name, params = verifier._normalize_grounded_action(skill_res.grounded_action)
        pddl_action = scene_graph.match_grounded_action(name, params)
        if pddl_action is None:
            print("Could not match pddl action!")
            break
        target = None
        match pddl_action.name.value:
            case "pickup_from" | "open" | "place_in" | "turn_on" | "turn_off":
                target = pddl_action.grounding[0].value
            case "place_on":
                target = pddl_action.grounding[2].value
                # TODO: table location is bad
        start_step = segment['start_step']
        end_step = segment['end_step']

        if target in scene_graph.object_data:
            target_object = scene_graph.object_data[target]
            target_info = target_object.to_dict(include_grounding=False)
            result = scene_graph.ground_openrouter(video_frames[start_step], target_object)
            if result['status'] == 'OK':
                target_info['image_point'] = result['position']
        else:
            target_info = None
        targets.append(target_info)
        scene_graph.apply_action(pddl_action)

    # NOTE: This may not contain the full trace, if verification fails...
    with gzip.open(f'{out_dir}/{episode_idx}_targets.json.zip', 'wt', encoding="ascii") as zipfile:
        json.dump(targets, zipfile)

In [ ]:
%env CC=/usr/bin/gcc
process_episode((reasonings, video_frames), 391)

In [ ]:
plt.close('all')
plt.figure(0)
plt.imshow(video_frames[0])
plt.scatter(*np.array([0.21289062, 0.66503906])*512)

In [ ]:
import re
import scipy
targets = []
target_names = []
scene_graph.reset_simulator()
subsample = 5
for segment in reasonings['segments']:
    skill = segment['skill']
    splits = re.split('([^a-zA-Z0-9]left[^a-zA-Z0-9]|[^a-zA-Z0-9]right[^a-zA-Z0-9])', skill)
    for i in range(1, len(splits), 2):
        if len(splits[i]) == 6:
            splits[i] = splits[i][0] + 'right' + splits[i][-1]
        elif len(splits[i]) == 7:
            splits[i] = splits[i][0] + 'left' + splits[i][-1]
    skill = ''.join(splits)
    skill_res = verifier.verify_skill(skill)
    print(skill_res)
    if not skill_res.accepted:
        print("Verification failed!")
        break
    name, params = verifier._normalize_grounded_action(skill_res.grounded_action)
    pddl_action = scene_graph.match_grounded_action(name, params)
    if pddl_action is None:
        print("Could not match pddl action!")
        break
    target = None
    match pddl_action.name.value:
        case "pickup_from" | "open" | "place_in" | "turn_on" | "turn_off":
            target = pddl_action.grounding[0].value
        case "place_on":
            target = pddl_action.grounding[2].value
            # TODO: table location is bad
    start_step = segment['start_step']
    end_step = segment['end_step']
    if target is not None:
        frame_start = start_step // subsample
        frame_end = end_step // subsample
        # Pad with an extra frame for interpolation, if we are not at end of sequence

        if frame_end <= frame_start:
            frame_end = frame_start + 1

        if frame_end < len(results) - 1:
            frame_end += 1
        def box_midpoint(box):
            return (box[0]+box[2]/2, box[1]+box[3]/2)
        frame_times = np.array(list(range(frame_start, frame_end))) * 5
        positions = np.array([box_midpoint(r[target].box_xywh) for r in results[frame_start:frame_end]])

        full_times = np.array(list(range(start_step, end_step)))
        positions_interp = scipy.interpolate.interp1d(frame_times, positions, axis=0, bounds_error=False, fill_value=(positions[0], positions[-1]))(full_times)
        targets.extend(positions_interp)
    else:
        targets.extend([None]*(end_step - start_step))
    target_names.extend([target] * (end_step - start_step))
        
    scene_graph.apply_action(pddl_action)


In [ ]:
plot_targets = np.array(targets)[:180]
plt.scatter(plot_targets[:, 0], -plot_targets[:, 1])
plt.xlim((0, 1))
plt.ylim((-1, 0))
plt.gca().set_aspect('equal')
plt.show()